In [ ]:
import PyPDF2
import re

def extract_text_from_pdf(file_path):
    """Extract text from PDF file"""
    try:
        with open(file_path, "rb") as file:
            reader = PyPDF2.PdfReader(file)
            text = ""
            for page in reader.pages:
                text += page.extract_text()
        return text.lower()
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found")
        return ""

# ATS-focused keyword categories
categories = {
    "skills": [
        "python", "java", "c", "c++", "machine learning",
        "deep learning", "sql", "numpy", "pandas", "javascript",
        "react", "node.js", "aws", "azure", "git", "docker"
    ],
    "education": [
        "b.tech", "b.e", "m.tech", "m.sc", "phd", "bachelor", "master",
        "university", "college", "gpa", "graduation"
    ],
    "experience": [
        "internship", "worked at", "years of experience", "project",
        "developed", "implemented", "designed", "managed", "led"
    ],
    "certifications": [
        "aws certified", "coursera", "udemy", "oracle certified",
        "certified", "certification"
    ],
    "soft_skills": [
        "communication", "teamwork", "leadership", "problem solving",
        "analytical", "critical thinking", "collaboration"
    ]
}

def check_ats_compatibility(text):
    """Check for ATS-unfriendly elements"""
    issues = []
    warnings = []
    
    # Check for formatting issues
    if re.search(r'[^\x00-\x7F]', text):  # Non-ASCII characters
        warnings.append("Contains non-ASCII characters - may not parse correctly in ATS")
    
    if text.count('\n') < 5:
        warnings.append("Very low line breaks - consider better formatting with bullet points")
    
    # Check for common ATS-friendly patterns
    if not re.search(r'\b(email|phone|linkedin|github)\b', text):
        issues.append("No contact information section detected")
    
    if not re.search(r'\b(experience|education|skills|projects)\b', text):
        warnings.append("Missing standard section headers (Experience, Education, Skills, Projects)")
    
    # Check for tables/complex formatting indicators
    if '|' in text and text.count('|') > 5:
        issues.append("Possible table formatting - ATS systems may not parse tables correctly")
    
    return issues, warnings

def analyze_resume(text, categories, required_skills=None):
    """Analyze resume for keyword matches and ATS compatibility"""
    results = {}
    found_skills = []
    
    # Extract keywords by category
    for category, keywords in categories.items():
        found = [kw for kw in keywords if kw in text]
        results[category] = found
        if category == "skills":
            found_skills.extend(found)
    
    # Calculate skill match percentage
    match = 0
    missing = []
    if required_skills:
        required_skills_lower = [s.lower() for s in required_skills]
        count = sum(1 for skill in required_skills_lower if skill in found_skills)
        match = (count / len(required_skills_lower)) * 100 if required_skills_lower else 0
        missing = [skill for skill in required_skills_lower if skill not in found_skills]
    
    # Check ATS compatibility
    ats_issues, ats_warnings = check_ats_compatibility(text)
    
    return results, match, missing, ats_issues, ats_warnings

def calculate_ats_score(results, ats_issues, ats_warnings, match_percentage):
    """Calculate overall ATS compatibility score"""
    score = 100
    
    # Deduct for missing issues
    score -= len(ats_issues) * 15
    
    # Deduct for warnings
    score -= len(ats_warnings) * 5
    
    # Add bonus for keyword variety
    total_keywords_found = sum(len(v) for v in results.values())
    score += min(total_keywords_found, 20)  # Max 20 bonus points
    
    # Factor in skill match
    score = (score + match_percentage) / 2
    
    return max(0, min(100, score))  # Ensure score is between 0-100

def generate_recommendations(results, ats_issues, ats_warnings, missing_skills):
    """Generate actionable recommendations"""
    recommendations = []
    
    # Critical issues
    if ats_issues:
        recommendations.append("\n  CRITICAL ISSUES TO FIX:")
        for issue in ats_issues:
            recommendations.append(f"  • {issue}")
    
    # Warnings
    if ats_warnings:
        recommendations.append("\n  FORMATTING IMPROVEMENTS:")
        for warning in ats_warnings:
            recommendations.append(f"  • {warning}")
    
    # Missing skills
    if missing_skills:
        recommendations.append("\n ADD MISSING REQUIRED SKILLS:")
        for skill in missing_skills[:5]:  # Show top 5
            recommendations.append(f"  • Add '{skill}' to your resume if applicable")
    
    # Low keyword categories
    recommendations.append("\n CATEGORY COVERAGE:")
    for category, keywords in results.items():
        coverage = len(keywords)
        if coverage == 0:
            recommendations.append(f"  • Add more {category} keywords (currently: 0)")
        elif coverage < 3:
            recommendations.append(f"  • Expand {category} section (currently: {coverage})")
    
    return recommendations

# ===== MAIN EXECUTION =====
# Run the analyzer 
resume_text = extract_text_from_pdf("Sudhanshu's Resume.pdf")

if resume_text:
    required_skills = ["python", "machine learning", "numpy", "pandas", "sql"]
    
    results, match, missing, ats_issues, ats_warnings = analyze_resume(
        resume_text, categories, required_skills
    )
    
    ats_score = calculate_ats_score(results, ats_issues, ats_warnings, match)
    recommendations = generate_recommendations(results, ats_issues, ats_warnings, missing)
    
    # Print comprehensive analysis
    print("=" * 60)
    print("ATS-FRIENDLY RESUME ANALYZER REPORT")
    print("=" * 60)
    
    print(f"\n ATS COMPATIBILITY SCORE: {ats_score:.1f}/100")
    
    print("\n RESUME CONTENT ANALYSIS")
    print("-" * 60)
    for category, found in results.items():
        count = len(found)
        print(f"  {category.replace('_', ' ').title():.<30} {count:>3} keywords found")
        if found and count <= 5:
            print(f"    Keywords: {', '.join(found)}")
    
    print(f"\n REQUIRED SKILLS MATCH: {match:.1f}%")
    if missing:
        print(f"  Missing: {', '.join(missing)}")
    
    print("\n" + "\n".join(recommendations))
    
    print("\n" + "=" * 60)
    print(" ATS BEST PRACTICES:")
    print("-" * 60)
    print("  • Use standard section headers (Experience, Education, Skills)")
    print("  • Include contact info (Email, Phone, LinkedIn)")
    print("  • Use bullet points for easy parsing")
    print("  • Stick to common fonts (Arial, Calibri, Times New Roman)")
    print("  • Avoid tables, images, and special formatting")
    print("  • Use standard keywords from job descriptions")
    print("  • Keep one-page format for easy scanning")
    print("=" * 60)
else:
    print("Failed to extract resume text. Please check the file path.")


Resume Analysis Summary
-----------------------
Skills found: ['c', 'c++']
Education found: []
Experience found: []
Certifications found: []
Soft_skills found: []

Skill Match Percentage: 0.00%
Missing Required Skills: ['python', 'machine learning', 'numpy', 'pandas']
